# Credit Card Balance Data Cleaning

This notebook cleans the monthly credit-card records using the problems found during EDA. It keeps records linked to the project applicants, groups rare contract statuses, flags unusual balance and payment values, and removes features with too many missing values.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
raw_path = project_root / "data" / "raw" / "credit_card_balance.csv"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "interim" / "credit_card_balance_clean.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [raw_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Raw input:", raw_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/credit_card_balance.csv
Clean output: /Users/taranveersingh/A-MRP/data/interim/credit_card_balance_clean.pkl


## Load data and retain project applicants


In [7]:
credit_raw = pd.read_csv(raw_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
project_ids = training_id_set.union(set(test_ids))
original_rows, original_columns = credit_raw.shape

credit_clean = credit_raw.loc[credit_raw["SK_ID_CURR"].isin(project_ids)].copy().reset_index(drop=True)
del credit_raw
out_of_scope_rows = original_rows - len(credit_clean)
print("Raw rows:", original_rows)
print("Project rows retained:", len(credit_clean))
print("Out-of-scope rows removed:", out_of_scope_rows)
print("Credit-card accounts represented:", credit_clean["SK_ID_PREV"].nunique())
print("Applicants represented:", credit_clean["SK_ID_CURR"].nunique())

Raw rows: 3840312
Project rows retained: 3227965
Out-of-scope rows removed: 612347
Credit-card accounts represented: 87452
Applicants represented: 86905


The rows that were removed belong to applicants outside the project's training and test data.


## Validate identifiers and monthly keys


In [10]:
missing_current_ids = int(credit_clean["SK_ID_CURR"].isna().sum())
missing_previous_ids = int(credit_clean["SK_ID_PREV"].isna().sum())
exact_duplicates = int(credit_clean.duplicated().sum())
duplicate_month_keys = int(credit_clean.duplicated(["SK_ID_PREV", "MONTHS_BALANCE"]).sum())
if exact_duplicates > 0:
    credit_clean = credit_clean.drop_duplicates().reset_index(drop=True)
assert missing_current_ids == 0 and missing_previous_ids == 0
assert duplicate_month_keys == exact_duplicates, "Conflicting records exist for the same account and month."
print("Missing applicant IDs:", missing_current_ids)
print("Missing previous-account IDs:", missing_previous_ids)
print("Exact duplicate rows removed:", exact_duplicates)
print("Conflicting account-month records:", duplicate_month_keys - exact_duplicates)

Missing applicant IDs: 0
Missing previous-account IDs: 0
Exact duplicate rows removed: 0
Conflicting account-month records: 0


The IDs are clean, no missing values, no duplicates, no conflicting records for the same account-month.


## Standardize contract status


In [13]:
credit_clean["NAME_CONTRACT_STATUS"] = credit_clean["NAME_CONTRACT_STATUS"].str.strip().fillna("Unknown")
training_mask = credit_clean["SK_ID_CURR"].isin(training_id_set)
status_counts = credit_clean.loc[training_mask, "NAME_CONTRACT_STATUS"].value_counts()
rare_statuses = status_counts[status_counts < 100].index
rare_status_mask = credit_clean["NAME_CONTRACT_STATUS"].isin(rare_statuses)
credit_clean.loc[rare_status_mask, "NAME_CONTRACT_STATUS"] = "Other rare"
print("Rare contract statuses consolidated:", list(rare_statuses))
credit_clean["NAME_CONTRACT_STATUS"].value_counts()

Rare contract statuses consolidated: ['Refused', 'Approved']


NAME_CONTRACT_STATUS
Active           3116673
Completed         100031
Signed              9939
Demand               843
Sent proposal        460
Other rare            19
Name: count, dtype: int64

Refused and Approved statuses are very rare here, these mostly belong to applications rather than active cards, so they get grouped into an Other rare category. Active and Completed still make up almost all of the records.


## Audit month, balance and payment behaviour


In [16]:
future_month = credit_clean["MONTHS_BALANCE"].gt(0)
extreme_month = credit_clean["MONTHS_BALANCE"].lt(-1200)
negative_balance = credit_clean["AMT_BALANCE"].lt(0)
negative_receivable = (
    credit_clean["AMT_RECEIVABLE_PRINCIPAL"].lt(0)
    | credit_clean["AMT_RECIVABLE"].lt(0)
    | credit_clean["AMT_TOTAL_RECEIVABLE"].lt(0)
)
negative_drawing = credit_clean["AMT_DRAWINGS_CURRENT"].lt(0)
balance_without_limit = credit_clean["AMT_BALANCE"].gt(0) & credit_clean["AMT_CREDIT_LIMIT_ACTUAL"].eq(0)
dpd_definition_above_dpd = credit_clean["SK_DPD_DEF"].gt(credit_clean["SK_DPD"])
payment_below_minimum = (
    credit_clean["AMT_PAYMENT_CURRENT"].notna()
    & credit_clean["AMT_INST_MIN_REGULARITY"].notna()
    & credit_clean["AMT_PAYMENT_CURRENT"].lt(credit_clean["AMT_INST_MIN_REGULARITY"])
)

credit_clean["CC_MONTH_ANOMALY"] = (future_month | extreme_month).astype("int8")
credit_clean["CC_NEGATIVE_BALANCE"] = negative_balance.astype("int8")
credit_clean["CC_NEGATIVE_RECEIVABLE"] = negative_receivable.astype("int8")
credit_clean["CC_NEGATIVE_DRAWING"] = negative_drawing.fillna(False).astype("int8")
credit_clean["CC_BALANCE_WITHOUT_LIMIT"] = balance_without_limit.astype("int8")
credit_clean["CC_DPD_INCONSISTENCY"] = dpd_definition_above_dpd.astype("int8")
credit_clean["CC_PAYMENT_BELOW_MINIMUM"] = payment_below_minimum.astype("int8")
credit_clean.loc[future_month | extreme_month, "MONTHS_BALANCE"] = np.nan
print("Future month values corrected:", int(future_month.sum()))
print("Extreme month values corrected:", int(extreme_month.sum()))
print("Negative balances retained and flagged:", int(negative_balance.sum()))
print("Negative receivable records retained and flagged:", int(negative_receivable.sum()))
print("Negative drawing records retained and flagged:", int(negative_drawing.sum()))
print("Positive balance with zero limit records:", int(balance_without_limit.sum()))
print("DPD definition inconsistencies:", int(dpd_definition_above_dpd.sum()))
print("Payments below minimum records:", int(payment_below_minimum.sum()))

Future month values corrected: 0
Extreme month values corrected: 0
Negative balances retained and flagged: 1967
Negative receivable records retained and flagged: 93850
Negative drawing records retained and flagged: 2
Positive balance with zero limit records: 6656
DPD definition inconsistencies: 0
Payments below minimum records: 106204


No future or extremely old month values were found. Some negative balances, negative receivable amounts, and a few negative drawing amounts were found. These are kept and flagged instead of removed, since they could be genuine corrections. A fair number of records also show a payment below the required minimum.


## Build training-only feature decisions


In [19]:
MISSING_THRESHOLD = 0.50
training_credit = credit_clean.loc[credit_clean["SK_ID_CURR"].isin(training_id_set)]
decision_rows = []
for column in credit_clean.columns:
    if column in ["SK_ID_CURR", "SK_ID_PREV"]:
        continue
    series = training_credit[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    decision = "Keep"
    reason = "Retain for applicant-level aggregation and later target-based selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-linked missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant among training-linked monthly records"
    decision_rows.append({
        "feature": column, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing), "decision": decision,
        "reason": reason, "target_association_stage": "After applicant-level aggregation"
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "missing_rate"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[
    feature_decisions["decision"] == "Remove", "feature"
] .tolist()
credit_clean = credit_clean.drop(columns=removed_features)
print("Features removed:", removed_features)
feature_decisions.round(5)

Features removed: ['CC_MONTH_ANOMALY', 'CC_DPD_INCONSISTENCY']


,feature,data_type,missing_count,missing_rate,unique_non_missing,decision,reason,target_association_stage
0,AMT_PAYMENT_CURRENT,float64,497918,0.19287,120031,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
1,AMT_DRAWINGS_ATM_CURRENT,float64,486683,0.18852,1885,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
2,AMT_DRAWINGS_OTHER_CURRENT,float64,486683,0.18852,1433,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
3,AMT_DRAWINGS_POS_CURRENT,float64,486683,0.18852,118733,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
4,CNT_DRAWINGS_ATM_CURRENT,float64,486683,0.18852,44,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
5,CNT_DRAWINGS_OTHER_CURRENT,float64,486683,0.18852,10,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
6,CNT_DRAWINGS_POS_CURRENT,float64,486683,0.18852,121,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
7,AMT_INST_MIN_REGULARITY,float64,212587,0.08235,240050,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
8,CNT_INSTALMENT_MATURE_CUM,float64,212587,0.08235,121,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
9,MONTHS_BALANCE,float64,0,0.00000,96,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation


Two features were removed: the month-anomaly and DPD-inconsistency flags created earlier in this notebook. Since no future/extreme months or DPD inconsistencies were actually found, both ended up constant.


## Fill categorical missingness and create missingness features


In [22]:
retained_categorical = credit_clean.select_dtypes(exclude="number").columns.tolist()
categorical_missing_before = int(credit_clean[retained_categorical].isna().sum().sum())
credit_clean[retained_categorical] = credit_clean[retained_categorical].fillna("Unknown")
record_features = [c for c in credit_clean.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]
credit_clean["CC_RECORD_MISSING_COUNT"] = credit_clean[record_features].isna().sum(axis=1).astype("int8")
credit_clean["CC_RECORD_MISSING_RATE"] = credit_clean["CC_RECORD_MISSING_COUNT"] / len(record_features)
print("Categorical missing values filled:", categorical_missing_before)
print(credit_clean[["CC_RECORD_MISSING_COUNT", "CC_RECORD_MISSING_RATE"]].describe().round(5))

Categorical missing values filled: 0
       CC_RECORD_MISSING_COUNT  CC_RECORD_MISSING_RATE
count             3.227965e+06            3.227965e+06
mean              1.481860e+00            5.699000e-02
std               3.032820e+00            1.166500e-01
min               0.000000e+00            0.000000e+00
25%               0.000000e+00            0.000000e+00
50%               0.000000e+00            0.000000e+00
75%               0.000000e+00            0.000000e+00
max               9.000000e+00            3.461500e-01


These two columns track how much information is missing for each monthly record. On average, a record is missing about 6% of its fields.


## Validate the cleaned table


In [25]:
numeric_columns = credit_clean.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(credit_clean[c].dropna()).sum()) for c in numeric_columns)
validation_checks = pd.DataFrame([
    {"check": "Only project applicants retained", "passed": set(credit_clean["SK_ID_CURR"]).issubset(project_ids)},
    {"check": "Applicant IDs complete", "passed": credit_clean["SK_ID_CURR"].notna().all()},
    {"check": "Previous-account IDs complete", "passed": credit_clean["SK_ID_PREV"].notna().all()},
    {"check": "Account-month keys unique", "passed": not credit_clean.duplicated(["SK_ID_PREV", "MONTHS_BALANCE"]).any()},
    {"check": "No categorical missing values", "passed": credit_clean.select_dtypes(exclude="number").isna().sum().sum() == 0},
    {"check": "No future balance months", "passed": not credit_clean["MONTHS_BALANCE"].gt(0).any()},
    {"check": "No negative delinquency days", "passed": not credit_clean[["SK_DPD", "SK_DPD_DEF"]].lt(0).any().any()},
    {"check": "No high-missing retained feature", "passed": not (feature_decisions.query("decision == 'Keep'")["missing_rate"] >= MISSING_THRESHOLD).any()},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one credit-card cleaning check failed."
validation_checks

,check,passed
0,Only project applicants retained,True
1,Applicant IDs complete,True
2,Previous-account IDs complete,True
3,Account-month keys unique,True
4,No categorical missing values,True
5,No future balance months,True
6,No negative delinquency days,True
7,No high-missing retained feature,True
8,No infinite numerical values,True


All checks passed.


## Save the clean table and audit reports


In [28]:
cleaning_audit = pd.DataFrame([
    {"rule": "Out-of-scope records removed", "affected": out_of_scope_rows},
    {"rule": "Exact duplicate rows removed", "affected": exact_duplicates},
    {"rule": "Rare status records consolidated", "affected": int(rare_status_mask.sum())},
    {"rule": "Negative balance records flagged", "affected": int(negative_balance.sum())},
    {"rule": "Negative receivable records flagged", "affected": int(negative_receivable.sum())},
    {"rule": "Payment-below-minimum records flagged", "affected": int(payment_below_minimum.sum())},
    {"rule": "Features removed by missingness/constant policy", "affected": len(removed_features)},
])
credit_clean.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "credit_card_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "credit_card_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "credit_card_cleaning_validation.csv", index=False)
print("Clean credit-card dataset saved:", output_path)
print("Output rows:", len(credit_clean))
print("Output columns:", credit_clean.shape[1])
print("Unique accounts:", credit_clean["SK_ID_PREV"].nunique())
print("Unique applicants:", credit_clean["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(credit_clean.select_dtypes(include="number").isna().sum().sum()))

Clean credit-card dataset saved: /Users/taranveersingh/A-MRP/data/interim/credit_card_balance_clean.pkl
Output rows: 3227965
Output columns: 30
Unique accounts: 87452
Unique applicants: 86905
Remaining numerical missing values: 4783385


## Main cleaning results

The cleaned credit-card balance data contains 3,227,965 monthly records for 87,452 credit-card accounts and 86,905 applicants. The IDs are complete and unique, and no duplicate records remain.

Rare contract statuses were grouped together, and unusual values, like negative balances or payments below the minimum, were kept and flagged instead of removed.

Two features were removed because they ended up constant. The cleaned data has 30 columns. The next step is to clean the POS/cash balance data.
